# 3.6 · 特征工程 / Feature Engineering

> **课程定位 / Where this fits**
> 第 6 课，**Part 3 · EDA 与数据预处理**。
> Lesson 6, **Part 3 · EDA & Preprocessing**.
>
> 有句老话："**数据和特征决定了机器学习的上限，模型只是逼近这个上限**"。特征工程就是用业务理解，从原始字段"造"出更有预测力的新特征。在表格数据竞赛和工业实战里，**好特征往往比调参更值钱**。
> An old saying: "**data and features set the ceiling of ML; models just approach it.**" Feature engineering uses domain understanding to *create* more predictive features from raw fields. On tabular data, **good features often beat hyperparameter tuning.**
>
> 💼 **实战/面试视角**："你会怎么做特征工程 / 怎么处理时间特征 / 怎么造交互特征" 是 case 面常客。
> 💼 **Practical/interview angle:** "how would you engineer features / handle datetime / build interactions" are case-interview staples.

> 💡 **面试相关 / Interview-relevant**
> - "怎么造有用的新特征 / 比率特征的价值"（出镜率 ★★★★）
> - "多项式/交互特征的维度爆炸"（★★★★）
> - "分箱怎么帮线性模型拟合非线性"（★★★★）
> - "时间/周期特征怎么编码（sin/cos）"（★★★★★）
> - "聚合特征（group-by）/ 防泄漏"（★★★★）

---

## 学习目标 / Learning Objectives

1. 用**比率特征**表达真实业务概念。
   Use **ratio features** to express real business concepts.
2. 理解**多项式/交互特征**及其维度爆炸。
   Understand polynomial/interaction features and their blow-up.
3. 用**分箱**让线性模型获得非线性表达力。
   Use **binning** to give linear models nonlinear expressiveness.
4. 提取**时间特征**并用 **sin/cos 周期编码**。
   Extract datetime features and use sin/cos cyclical encoding.
5. 构造 **group-by 聚合特征**（并注意泄漏）。
   Build group-by aggregation features (and watch for leakage).

## 目录 / TOC
1. [先建直觉 + 数据](#1)
2. [比率特征 ⭐](#2)
3. [多项式/交互特征 ⭐](#3)
4. [分箱：驯服非线性 ⭐](#4)
5. [时间特征 + 周期编码 ⭐](#5)
6. [聚合特征 ⭐](#6)
7. [实战对比 + 小结](#7)


<a id="1"></a>
## 1. 先建直觉 + 数据 / Intuition & Data

特征工程的核心思路：**把"模型不容易直接学到的东西"显式地喂给它**。模型（尤其线性模型）学不会"人均房间数 = 房间数 ÷ 人数"这种组合，但你可以**直接算好这一列**交给它。
The core idea: **explicitly hand the model things it can't easily learn on its own.** A model (especially a linear one) won't discover "rooms-per-person = rooms ÷ people", but you can **precompute that column** and feed it in.

几类最常用的特征工程：**比率**（除法组合）、**交互/多项式**（乘法组合）、**分箱**（连续→离散）、**时间特征**（从时间戳提取）、**聚合**（按某键 group-by 统计）。我们用 California Housing 逐一演示。
The most common types: **ratios** (division), **interactions/polynomials** (multiplication), **binning** (continuous→discrete), **datetime features**, and **aggregations** (group-by stats). We demo each on California Housing.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

data = fetch_california_housing(as_frame=True)
df = data.frame.copy()
print(f"shape: {df.shape}")
print("列 columns:", df.columns.tolist())
df.head(3)


<a id="2"></a>
## 2. 比率特征 ⭐ / Ratio Features

**比率特征**（两列相除）常常比原始列更有意义，因为它表达了一个**真实概念**。例如 `AveRooms`（平均房间数）单看意义有限，但 `AveRooms / AveOccup` = **人均房间数 = 拥挤程度**，这个概念和房价关系更直接。
**Ratio features** (dividing two columns) are often more meaningful than the raw columns because they express a **real concept**. E.g. `AveRooms` alone is limited, but `AveRooms / AveOccup` = **rooms per person = crowding**, a concept more directly tied to house value.


In [ ]:
# 比率特征: 用除法表达"人均房间(拥挤度)"和"卧室占比" / ratio features = real concepts
df["rooms_per_person"] = df["AveRooms"] / df["AveOccup"]    # 人均房间 = 拥挤度
df["bedrooms_ratio"]   = df["AveBedrms"] / df["AveRooms"]    # 卧室占总房间的比例

# 验证新特征是否比原始列更相关 / does the ratio correlate more with the target?
target = df["MedHouseVal"]
print("各特征与目标(房价中位数)的相关 correlation with target:")
for col in ["AveRooms","AveOccup","rooms_per_person","bedrooms_ratio"]:
    print(f"  {col:<20} corr = {df[col].corr(target):+.3f}")
print("\nrooms_per_person 比原始 AveRooms 更相关 → 比率抓住了'拥挤度'这个真实概念")


<a id="3"></a>
## 3. 多项式/交互特征 ⭐ / Polynomial & Interaction Features

**交互特征**（两列相乘）捕捉"两个变量合起来的效应"（如收入 × 房龄）。**多项式特征**还包括平方、立方等。`PolynomialFeatures` 一键生成，但要警惕**维度爆炸**：$d$ 个特征做 2 阶多项式会产生 $\sim d^2/2$ 个新特征。实战建议：**只对少数关键特征做交互**，不要全量。
**Interaction features** (multiplying two columns) capture the joint effect (income × age). **Polynomial features** also add squares, cubes, etc. `PolynomialFeatures` generates them in one call, but beware the **blow-up**: $d$ features at degree 2 give $\sim d^2/2$ new ones. Practical advice: **interact only a few key features**, not all.


In [ ]:
from sklearn.preprocessing import PolynomialFeatures

# 维度爆炸演示: 特征数翻倍式增长 / dimension explosion
for d in [5, 10, 50]:
    n_out = PolynomialFeatures(degree=2, include_bias=False).fit_transform(np.zeros((1, d))).shape[1]
    print(f"{d:>2} 个特征 → 2阶多项式 → {n_out} 个特征")
print("→ 高维下慎用; 通常只对少数关键特征做\n")

# 实用做法: interaction_only=True 只要交叉项(x1*x2), 不要纯平方项(x1²) / interaction-only
pf = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
pf.fit(df[["MedInc","HouseAge"]])
print(f"MedInc × HouseAge 的交互特征: {list(pf.get_feature_names_out(['MedInc','HouseAge']))}")


<a id="4"></a>
## 4. 分箱：驯服非线性 ⭐ / Binning: Taming Nonlinearity

**分箱(binning)** 把连续特征切成若干区间，再 One-Hot。它的妙用：**让线性模型也能拟合非线性关系**。因为分箱后每个区间有独立的系数，模型可以拼出"分段常数"曲线，从而逼近 U 形等非线性。
**Binning** cuts a continuous feature into intervals, then One-Hots them. Its magic: **it lets even a linear model fit nonlinear relationships.** Each bin gets its own coefficient, so the model can build a piecewise-constant curve and approximate U-shapes, etc.


In [ ]:
from sklearn.preprocessing import KBinsDiscretizer

# 构造一个 U 形关系: y = (x-5)², 直线根本拟合不了 / a U-shaped relationship
x = rng.uniform(0, 10, 2000)
y_u = (x - 5)**2 + rng.normal(0, 3, 2000)

# 线性模型直接用原始 x → 失败(直线拟合不了 U) / linear on raw x fails
lr_raw = cross_val_score(LinearRegression(), x.reshape(-1,1), y_u, cv=5, scoring="r2").mean()

# 把 x 分成 10 个箱再 one-hot → 线性模型能逼近 U 了 / bin + onehot, then linear succeeds
binner = KBinsDiscretizer(n_bins=10, encode="onehot-dense", strategy="quantile")
x_binned = binner.fit_transform(x.reshape(-1,1))
lr_binned = cross_val_score(LinearRegression(), x_binned, y_u, cv=5, scoring="r2").mean()

print("U 形关系 y=(x-5)²:")
print(f"  线性模型 + 原始 x: R² = {lr_raw:.3f}  (失败 — 一条直线拟合不了 U)")
print(f"  线性模型 + 分箱 x: R² = {lr_binned:.3f}  (成功! 每箱一个系数, 拼出 U 形)")
print("分箱给了线性模型'分段常数'的表达力 → 驯服了非线性")


<a id="5"></a>
## 5. 时间特征 + 周期编码 ⭐ / Datetime & Cyclical Encoding

时间戳本身没法直接喂模型，要**拆成日历成分**：年、月、星期几、小时、是否周末等。
A timestamp can't be fed directly; **decompose it into calendar components**: year, month, day-of-week, hour, is-weekend, etc.

更关键的是**周期编码**：小时是循环的——23 点和 0 点其实**相邻**，但数值上差 23。直接用数值会骗模型"它们差很远"。用 **sin/cos** 把它映射到一个圆上，首尾就自然相接了。这是时间特征的高频考点。
The key subtlety is **cyclical encoding**: hours are circular — 23:00 and 00:00 are actually **adjacent**, but numerically differ by 23. Raw numbers fool the model into "they're far apart". Mapping to a circle via **sin/cos** makes the ends meet. A frequently-tested point.


In [ ]:
# 造一批随机时间戳并拆出日历成分 / random timestamps -> calendar components
ts = pd.to_datetime("2026-01-01") + pd.to_timedelta(rng.integers(0, 365*24*3600, 3000), unit="s")
tsdf = pd.DataFrame({"ts": ts})
tsdf["month"]       = tsdf.ts.dt.month
tsdf["day_of_week"] = tsdf.ts.dt.dayofweek          # 0=周一 Monday
tsdf["hour"]        = tsdf.ts.dt.hour
tsdf["is_weekend"]  = (tsdf.ts.dt.dayofweek >= 5).astype(int)   # 周六日=1
print(tsdf.head(3))

# 周期编码: 把 hour 映射到单位圆上的 (sin, cos) / map hour onto a circle
tsdf["hour_sin"] = np.sin(2*np.pi*tsdf.hour/24)
tsdf["hour_cos"] = np.cos(2*np.pi*tsdf.hour/24)
def hr_vec(h): return np.array([np.sin(2*np.pi*h/24), np.cos(2*np.pi*h/24)])
print(f"\n原始数值: |23 - 0| = 23 (模型以为差很远)")
print(f"周期编码后: ||23点-0点|| = {np.linalg.norm(hr_vec(23)-hr_vec(0)):.3f}, ||23点-22点|| = {np.linalg.norm(hr_vec(23)-hr_vec(22)):.3f}")
print("→ 23点和0点的距离 ≈ 23点和22点 — 首尾相接成立! ✓")

fig, ax = plt.subplots(figsize=(4.5, 4.5))
hours = np.arange(24)
ax.scatter(np.sin(2*np.pi*hours/24), np.cos(2*np.pi*hours/24), s=80)
for h in hours: ax.annotate(str(h), hr_vec(h)*1.12, ha="center", fontsize=8)
ax.set_aspect("equal"); ax.set_title("hour 周期编码: 24 个点排成一个圆\n0 点和 23 点相邻 (cyclical)")
plt.tight_layout(); plt.show()


<a id="6"></a>
## 6. 聚合特征 ⭐ / Aggregation Features

**聚合(group-by)特征**：按某个键（用户、商品、地区）统计历史行为，是推荐、风控、用户画像的核心武器。例如"该用户的历史平均消费额"，再派生出"本次消费相对其均值的偏离"——这正是**异常/欺诈检测**的强特征（接 5.14）。
**Aggregation (group-by) features:** summarize historical behavior per key (user, item, region) — the core weapon of recommendation, risk, and user profiling. E.g. "a user's average past spend", then "this transaction relative to their norm" — a strong **anomaly/fraud** feature (see 5.14).

> ⚠️ **泄漏警告**：聚合特征极易泄漏！如果用"包含未来的全量历史"算用户均值，就把未来信息泄漏进了当前预测。生产里必须用**截至当前时刻**的历史（时间窗口聚合），并且统计量只在训练集上算。
> ⚠️ **Leakage warning:** aggregation features leak easily! Computing a user mean over data that includes the future leaks future info into the present. In production, aggregate over history **up to the current moment** (time-windowed), and compute stats on train only.


In [ ]:
# 模拟用户交易, 构造每用户的聚合统计 / per-user aggregation features
trans = pd.DataFrame({"user_id": rng.integers(1, 100, 2000), "amount": rng.lognormal(3, 1, 2000)})
# groupby + agg: 一次算出每个用户的多个统计量 / multiple stats per user in one call
user_stats = trans.groupby("user_id")["amount"].agg(
    user_mean="mean", user_std="std", user_count="count", user_max="max").round(2)
trans = trans.merge(user_stats, on="user_id")        # 把用户级统计 merge 回每行
# 派生特征: 本次金额相对该用户均值的倍数 → 异常交易信号 / deviation from the user's norm
trans["amount_vs_user_mean"] = trans["amount"] / trans["user_mean"]
print("聚合特征示例 aggregation features:")
print(trans.head(4).round(2).to_string(index=False))
print("\namount_vs_user_mean > 3 → 远超该用户平时 → 欺诈/异常信号 (5.14)")


<a id="7"></a>
## 7. 实战对比 + 小结 / Showdown & Summary

把工程特征加进去，看模型表现提升多少。这里给 California Housing 加几个**地理特征**（到洛杉矶/旧金山的距离——沿海大城市附近房价高）。
Add the engineered features and measure the lift. Here we add **geographic features** to California Housing (distance to LA/SF — coastal big cities raise prices).


In [ ]:
from sklearn.ensemble import RandomForestRegressor

base_cols = ["MedInc","HouseAge","AveRooms","AveBedrms","Population","AveOccup","Latitude","Longitude"]
# 地理特征: 到 LA/SF 的距离, 取较近者作"离大城市的距离" / distance-to-city features
la, sf = (34.05, -118.24), (37.77, -122.42)
df["dist_LA"] = np.sqrt((df.Latitude-la[0])**2 + (df.Longitude-la[1])**2)
df["dist_SF"] = np.sqrt((df.Latitude-sf[0])**2 + (df.Longitude-sf[1])**2)
df["dist_city"] = df[["dist_LA","dist_SF"]].min(axis=1)
eng_cols = base_cols + ["rooms_per_person","bedrooms_ratio","dist_LA","dist_SF","dist_city"]

rf = RandomForestRegressor(n_estimators=50, random_state=0, n_jobs=-1)
r2_base = cross_val_score(rf, df[base_cols], df.MedHouseVal, cv=3, scoring="r2").mean()
r2_eng  = cross_val_score(rf, df[eng_cols], df.MedHouseVal, cv=3, scoring="r2").mean()
print(f"基线(8 原始特征) baseline:        R² = {r2_base:.3f}")
print(f"+ 比率 + 地理距离特征 engineered:   R² = {r2_eng:.3f}")
print(f"提升 lift +{(r2_eng-r2_base)*100:.1f} 个百分点 — 几个好特征胜过调参\n")

rf.fit(df[eng_cols], df.MedHouseVal)
imp = pd.Series(rf.feature_importances_, index=eng_cols).sort_values(ascending=False)
print("特征重要性 Top 5 feature importance:")
print(imp.head(5).round(3).to_string())


```
特征工程 = 把模型难直接学的概念显式喂给它; 好特征 > 调参
比率特征: 两列相除表达真实概念(人均房间=拥挤度)
多项式/交互: 乘法组合; PolynomialFeatures 维度爆炸 → 只对关键特征做
分箱: 连续→区间→one-hot, 让线性模型获得分段表达力, 拟合非线性
时间特征: 拆日历成分(月/星期/小时/周末); 周期量(小时/月)用 sin/cos 编码(首尾相接)
聚合特征: group-by 统计(用户均值等) → 推荐/风控核心; ⚠ 极易泄漏(别含未来), 时间窗口+只train算
```

### 💡 面试速查 / Interview cheat-sheet
1. **比率特征**常比原始列更有意义（表达真实概念）。
   Ratio features often beat raw columns (express real concepts).
2. **多项式/交互会维度爆炸**，只对关键特征做。
   Polynomial/interactions blow up dimensions; apply selectively.
3. **分箱让线性模型拟合非线性**（分段常数表达力）。
   Binning lets linear models fit nonlinearity (piecewise-constant).
4. **周期特征用 sin/cos**（23点和0点相邻）。
   Cyclical features use sin/cos (23:00 next to 00:00).
5. **聚合特征强但极易泄漏**：时间窗口 + 只在训练集算。
   Aggregations are powerful but leak easily: time-windowed + train-only.

### 下一节 / Next
**3.7 文本特征工程**——文字怎么变成数字特征：词袋、TF-IDF、n-gram，以及它们和后面 NLP 的衔接。
**3.7 Text Features** — turning text into numeric features: bag-of-words, TF-IDF, n-grams, bridging to NLP later.
